# Lab Exercise: Building Energy Atlas @Uni Graz with OSM + GHS-OBAT

**📝 Scenario.** The University of Graz Estates & Sustainability Office needs a *rapid, transparent, open-data proxy* to estimate and visualise building heating demand within **1 km** of campus. The output is used for **triage** (deciding which buildings deserve a detailed audit first), not for final retrofit design.

**🚚 What you will deliver**
1. A reproducible notebook that:
   - downloads building footprints from OpenStreetMap (OSM)
   - loads a subset of **GHS-OBAT** building attributes (epoch, height, use)
   - derives geometry/volume features with **explicit source tracking**
   - computes energy demand with **low/mid/high scenarios**
   - produces a Plotly map + a ranked table of candidates

2. A short advisory brief (<500 words) answering:
   - **top 3 buildings to prioritise** (based on your computed evidence and assumptions)
   - **technical + ethical risks** of relying on OSM + OBAT for policy
   - a **basic governance protocol** (“how to use this responsibly”)


#### 👀 Engineering & Academic Quality
- **Correctness:** geometry handling, CRS, joins, units, sanity checks
- **Engineering:** modular functions, readable code, minimal repetition, assertions/tests
- **Reasoning:** assumptions documented; uncertainty quantified; limitations not hidden
- **Communication:** map + brief answer the client question directly

<div class="alert alert-warning">

#### 📃 Data sources & licences (you must always acknowledge them)
- OpenStreetMap is open data licensed under the ODbL. [👀 see]( https://www.openstreetmap.org/copyright)
- GHS-OBAT is published by the European Commission JRC and provides building-level attributes (height, epoch, use, compactness) linked to Overture building footprints under the license Creative commons (CC BY 4.0)

<div class="alert alert-info">  

#### 🔗 Useful links:
- GHS data catalogue ([website](https://human-settlement.emergency.copernicus.eu/downloadWizard.php))
- GHS-OBAT paper ([pdf](https://www.sciencedirect.com/science/article/pii/S2352340925004780))
- TABULA Austria scientific report ([pdf](https://episcope.eu/fileadmin/tabula/public/docs/scientific/AT_TABULA_ScientificReport_AEA.pdf)) (typologies / construction periods)
- TABULA online ([website](https://webtool.building-typology.eu/#bm))

---

## 🎯 Learning outcomes

---

1. **Acquire** vector data programmatically (OSM) via `OSMnx` and **load** external geospatial data (OBAT subset) using `requests`
2. **Control spatial reference systems (CRS)** so that area/volume calculations have correct units
3. **Design and evaluate a spatial join/conflation strategy** (not just “sjoin and pray”)
4. **Engineer features** (area, floors, heated volume) with explicit assumptions and missing-data strategy
5. **Build a simple, transparent proxy model** and communicate uncertainty and data ethics
6. **Package work like a software engineer**: config, functions, tests, and reproducible outputs

> If you only produce a map but cannot explain the join logic and assumptions, you have not met the brief

<div class="alert alert-warning">     

**⚠️ Important:** If you use generative AI to **_help_** you with this exercise, you must use the prompt present in `AGENTS.md` to start your session.

---

## 0. 🔧 Setup

---


### 0.1 Get the course materials

Clone or download the course materials from the repository.
```sh
git clone <repository-url>
```

You can also update the repository with the latest changes (only after you have cloned it):
```sh
git pull # do a git pull at the beginning of each lecture to get the most recent materials or use the pull button on VScode
```

<div class="alert alert-info">  

Try to always start your Jupyter notebooks with all necessary imports in the first cell. This is good practice: 
- better readability
- easier to debug
- easier to share

In the following cells, record all parameters you use in your analysis (urls, paths, constants, CRS codes, etc.). This will help you to reproduce and tweak your work later.

**Good practice**: separate environment-specific settings (paths, credentials) into a `.env` file, and analysis constants (thresholds, CRS, buffer sizes) into a `config.yaml` or `config.py`. Never hardcode absolute paths — use `pathlib.Path` with relative paths so your notebook runs on any machine.

### 0.2 Imports and Project structure

In [ ]:
# installs for the notebook, uncomment to install them if not done already, or select the library you need to install
# %uv add numpy pandas geopandas osmnx plotly shapely

In [8]:
# If you are missing packages, install them once in your environment, not every run.

# system
from pathlib import Path
import io
import os

# dowdload
import requests
import zipfile

# geo
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox

# viz
import plotly.express as px

In [ ]:
# `./` represents the current directory and `../` the parent directory

# Project structure (adjust if needed)
DATA_DIR = Path("./data") # where you will download your raw data
OUTPUT_DIR = Path("./outputs") # where you will save your outputs
DATA_DIR.mkdir(exist_ok=True) # create directory if it does not exist
OUTPUT_DIR.mkdir(exist_ok=True) # create directory if it does not exist

<div class="alert alert-info">  

Look at your folder structure, you should have a `data` and an `outputs` folder

### 0.3 Configuration

<div class="alert alert-info">  

In software engineering, “magic” numbers or variables scattered across cells are a maintenance bug.  
Put assumptions and parameters in a config dictionary so they are easy to audit.

In [ ]:
CONFIG = {}

# add elements to the CONFIG dictionary
CONFIG["radius"] = 1000 # m 
CONFIG["crs_geographic"] = "EPSG:4326" 
CONFIG["crs_projected"] = "EPSG:32633" 

CONFIG["default_floor_height_m"] = 3.0   # used if only floor number is known
CONFIG["default_levels"] = 2             # used if neither height nor levels are available

CONFIG


---

## 1. 🌍 Define the Area of Interest (AOI)

---

We work with a **1 km radius** around the campus point.

**Workflow:** 
1. get the coordinate of Uni Graz
2. build a point in WGS84
3. project to metres
4. buffer
5. keep the buffered polygon

<div class="alert alert-warning">

You can’t use degrees as if they were meters. In a geographic CRS (like EPSG:4326), coordinates are in degrees of latitude/longitude, and the size of one degree changes with latitude. So a “buffer of 0.01” degrees is not a fixed real‑world distance and will be different near the equator than near the poles.
To get a meaningful distance buffer (e.g. 500 m), first reproject your data to a projected CRS in meters (e.g. a suitable UTM zone) and then buffer in that CRS.

Our Tender mentions the University of Graz and a radius of 1km around it. 

We need to determine the coordinates of the University of Graz. We therefore need to use a geocoder to transform the place name into coordinates.
We will use the geocoding function from the `osmnx` library.

Look at the documentation of the `geocode` function from the [`osmnx`](https://osmnx.readthedocs.io/en/stable/user-reference.html) library.

<div class="alert alert-info">

**Geocoding** means converting a human description of a place (like an address or place name) into geographic coordinates (latitude/longitude) by matching it against a reference database of locations.

<div class="alert alert-danger">     

**🚀 TODO**
- Implement the geocoding function here (use osmnx.geocode)
- Integrate the geocoded coordinates into the config dictionary lat/lon

<div class="alert alert-success"> 

**🧠 Questions**: 
- How to verify your coordinates? 
- Did you geocode the good address? 

<div class="alert alert-danger">     

**🚀 TODO**

Write a function that uses the coordinates to create a buffer of 1km 
1. create a point geometry from the coordinates
2. create a GeoDataFrame from the point
3. reproject the point to the projected CRS
4. buffer the point
5. create a GeoDataFrame from the buffer
6. reproject the buffer to WGS84

In [ ]:
def make_aoi_polygon(lat: float, lon: float, radius_m: float, crs_proj: str) -> gpd.GeoDataFrame:
    """Create a simple buffer around a point"""
    
    return gdf_buffer_wgs84

<div class="alert alert-info">

**💡 Tips:**
- Use the already imported `shapely` library to create a [point](https://shapely.readthedocs.io/en/2.1.2/reference/shapely.Point.html)
- Use `geopandas` to change the CRS and create a buffer



<div class="alert alert-danger">     

**🚀 TODO**

Verify that the area is coherent
1. calculate the area of the buffer (π·R²)
2. AOI area is ~ π·1000² ≈ 3.14 km².

---

## 2. 🏘️ Download building footprints from OpenStreetMap via OSMnx

---

We use OSMnx to query OSM features within the AOI. OSMnx provides a convenient API for retrieving OSM geometries and attributes.

<div class="alert alert-info">

**Engineering note:** treat external APIs as unstable. Cache results locally (GeoPackage) so re-running your notebook is deterministic.


<div class="alert alert-danger">     

**🚀 TODO**

Load osm buildings from OSMnx
1. check if the CRS of your buffer is in 4326 
2. define the right set of OSM tags to get buildings geometries (you can add the tags within the CONFIG)
3. download buildings within the AOI in a geodataframe using the appropriate function of osmnx


<div class="alert alert-info">

**💡 Tips:**
- with the last version of OSMnx you can use ox.features_from_polygon(...)

### 2.1 Cache and reload

If you successfully downloaded the OSM buildings once, save them.  
On rerun, load from disk, it will make your script faster and more robust!


<div class="alert alert-info">

**Caching** stores copies of data (like API responses or external downloads) locally so you can reuse it without refetching every time. This helps boost speed and makes your code more robust against network failures or API downtime.

**But you must invalidate/update the cache when source data changes, or you'll work with stale information.**

<div class="alert alert-danger">     

**🚀 TODO**

Cache OSM buildings
1. create a path for the file to store the building
2. save/load the data in [geoparquet](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_parquet.html)
3. Update your previous div: check if the file exist, if this is the case load the file instead of downloading the data

### 2.2 Minimal cleaning

When you download buildings from OpenStreetMap, the raw GeoDataFrame often contains a mix of geometry types, invalid shapes, and missing or empty geometries. Clean these first before doing any area, height, or join operations.


<div class="alert alert-danger">     

**🚀 TODO**

Create a function that cleans the geometries:
1. keep Polygon / MultiPolygon 
2. fix invalid geometries 
3. drop empty geometries
4. project to projected CRS for area computations
5. run the function on the buildings

In [ ]:
def clean_buildings(gdf: gpd.GeoDataFrame, crs_proj: str) -> gpd.GeoDataFrame:
    """Basic geometry cleaning for building footprints."""

    return gdf



<div class="alert alert-info">

**💡 Tips:**
- Filter rows with a Boolean condition (or mask)
    ```python
        gdf = gdf[condition] # This keeps only the rows where condition is True
    ```
- Check the geometry type 
    ```python
        gdf.geometry.geom_type.value_counts() # This inspects geometry types, do you have other types than polygons and multipolygons?
        gdf.geometry.geom_type.isin(["GEOMETRY_TYPES"]) # You can build a filter condition 
    ```
- Remove missing and empty geometries
    1. missing: the geometry is None / NaN: `dropna()` or via filter condition `gdf.geometry.notna()`
    2. empty: the geometry exists, but contains no actual shape: `gdf.geometry.is_empty`
- Repair geometries (self-intersections or topology problems)
    1. Prefer `make_valid()` when available, because it is more explicit and generally more robust (see [shapely documentation](https://shapely.readthedocs.io/en/2.1.2/reference/shapely.make_valid.html))
    2. Use `buffer(0)` as a fallback or quick fix, but remember that it may slightly alter geometries


<div class="alert alert-success"> 

**🧠 Questions**:
- What percentage of downloaded features were not polygons or invalid?
- Is it good enough to drop invalid geometries?

### 2.3 Visualize your results 

Always visualize your results to understand the data and the model. Human eyes is the best tool to spot patterns and anomalies.

<div class="alert alert-danger">     

**🚀 TODO**

Plot and verify your results
1. plot the OSM buildings 
2. plot your buildings within Graz area
3. verify if the buildings are correctly plotted

<div class="alert alert-info">

**💡 Tips:**
- you can use the default [`plot()`](https://geopandas.org/en/stable/docs/user_guide/mapping.html) method that comes with all geodataframes. Under the hood, it uses [matplotlib](https://matplotlib.org/).
- you can get the geometry of Graz area using OSMnx
- to plot multiple layers, you can add the `ax` parameter to the `plot()` method of each layer.

---

## 3. 🗼 Download & Load GHS-OBAT attributes

---

GHS-OBAT provides building-level attributes for all buildings in the world. The dataset includes:

- `height` (mean building height, metres)
- `use` (0=outside domain, 1=residential, 2=non-residential)
- `epoch` (construction period bins)
- `shapefactor` (compactness proxy)
- `area`, `perimeter` (computed in local UTM)

OBAT is massive and is distributed in country/area chunks.
We will focus on the data in Austria.

### 3.1 Download dynamically OBAT data

<div class="alert alert-danger">     

**🚀 TODO**

Collect OBAT data 
1. identify the correct URL of the data (choose the csv format)
2. download & extract the data via the request library in the `data_raw` folder (use the provided function)
3. load the data into a DataFrame
4. reproject the OBAT data to the AOI CRS (WGS84 = EPSG:4326)
5. save the data to a parquet 
6. add a test before the download line to check if the file already exists
 

In [11]:
def download_zip(url: str, output_dir: str):
    """Download a zip file and extract it to the specified directory."""
    r = requests.get(url, timeout=30)
    r.raise_for_status() # raise an exception if the request failed
    with zipfile.ZipFile(io.BytesIO(r.content)) as zip_ref:
        zip_ref.extractall(output_dir)

<div class="alert alert-info">

**💡 Tips:**
- You can load a csv file with `pd.read_csv()`

<div class="alert alert-success"> 

**🧠 Questions**:
- What differences do you observe between the csv and the Parquet format?

### 3.2 📊 Inspect schema and CRS

<div class="alert alert-danger">     

**🚀 TODO**

Inspect the OBAT data
- columns
- data types
- CRS
- row count

### 3.3 📏 Clip OBAT to the AOI



<div class="alert alert-danger">     

**🚀 TODO**

Clip the OBAT layer to the AOI
1. Create point geometries for the OBAT table from its longitude and latitude columns
2. control and set the crs
2. (if needed) reproject the OBAT layer to the same CRS as the AOI
3. clip the OBAT layer to the **extent** of the AOI
4. Save the clipped layer to the `data` folder

<div class="alert alert-info">

**Why do we clip early?** 

In spatial data science, clipping early is a good engineering decision because it:

- reduces the number of features you process
- makes spatial joins faster
- lowers memory use
- makes debugging easier
- keeps the workflow focused on the study area

This is a common optimisation step in reproducible geospatial pipelines: reduce the data as soon as you know the valid spatial scope of the analysis.

**Why use the AOI extent instead of the AOI geometry?**

Here we clip OBAT points to the **bounding box** (extent) of the AOI, not directly to the circular AOI polygon.

- it is simpler and often faster
- it avoids losing borderline points because of small geometric precision issues
- it keeps a small safety margin around the AOI

_Important trade-off_

Using the extent is **less precise** than using the AOI geometry itself:
- it may keep some points that are outside the circular AOI
- so it is best understood as a **first coarse crop**

If your final analysis requires strict membership in the AOI, you can apply a second, more precise spatial filter afterwards.


<div class="alert alert-info">

**💡 Tips:**

- **Create geometries from longitude and latitude** If your OBAT table is still a normal pandas DataFrame, convert it into a GeoDataFrame first:

```python
    obat = gpd.GeoDataFrame(
        obat,
        geometry=gpd.points_from_xy(obat["lon"], obat["lat"]),
        crs="EPSG:4326"
    )
```

- **AOI extent** The AOI extent can be accessed with: `aoi.total_bounds`
- **Clipping operation** Use the [`clip`](https://geopandas.org/en/stable/docs/reference/api/geopandas.clip.html) method from GeoPandas to clip the OBAT layer to the AOI. (**💪 Pro Tip** you can also use [`cx`](https://geopandas.org/en/v1.1.0/docs/reference/api/geopandas.GeoDataFrame.cx.html))


<div class="alert alert-success"> 

**🧠 Questions**:
- How many OBAT points are there before clipping and after clipping?
- How much did clipping reduce the dataset?
- Which step would fail or become misleading if the CRS of OBAT and AOI were different?

### 3.4 Visualize your results 

<div class="alert alert-danger">     

**🚀 TODO**

Plot and verify your results
1. plot the OBAT data 
2. overlap the OBAT points on the OSM buildings

---

## 4. 🔗 Spatial Join OSM buildings to OBAT attributes

---

##### Why this join is non-trivial

- OSM building geometries come from a volunteered mapping community.
- GHS-OBAT attributes are linked to **Overture** building footprints, not directly to OSM footprints. [Overture buildings](https://docs.overturemaps.org/guides/buildings/) are derived largely from OSM data, but also completed and corrected using AI-based methods, so geometries will not match perfectly. In Austria, OSM quality is generally high, so we might be lucky in practice.

We are therefore **not** doing a simple “clean spatial join”. We are doing **geospatial conflation**: matching _approximate_ geometries across different sources and accepting that some links will be wrong or missing.

Your task is to:
1. choose a matching rule
2. quantify its failure modes (where and how it mis-matches or misses buildings)
3. carry this uncertainty forward into your brief (state clearly how reliable your results are).


### 4.1 Matching options

**Option 1 — Centroid-in-polygon join**  
Match an OSM footprint to the OBAT feature whose centroid falls inside the OSM polygon.

**Option 2 — Nearest-neighbour join**  
Match by nearest centroid within a distance threshold.

For this lab exercise, we will implement **Option 1**. You can try **Option 2** as a bonus exercise.

<div class="alert alert-danger">     

**🚀 TODO**

Join your OSM data with the OBAT data.
1. choose a join method `predicate` (intersects / contains / within / touches / crosses / overlaps)
2. join the data (use `sjoin` from `geopandas`)
3. (optional but recommended) create a new column `has_obat` to indicate if the building has OBAT data
4. visualize the result, highlight buildings with missing OBAT values

<div class="alert alert-info">

**💡 Tips:**

- to make a spatial join between two GeoDataFrames, use [`sjoin()`](https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html), you will have to define the `how` and `predicate` parameters. Try different combinations to see what works best.

### 4.2 Match-quality report

<div class="alert alert-danger">     

**🚀 TODO**

Create a brief summary print with:
1. % of OSM buildings with a match
2. potential of missing OSM buildings (compared to OBAT) within the AOI 
3. OBAT is providing an area column, compare this area with the OSM area
4. identify potential issues (e.g. several OBAT buildings within the same OSM building)

<div class="alert alert-success"> 

**🧠 Questions**:
- Which error is more dangerous for policy: (a) leaving buildings unmatched, or (b) matching the wrong OBAT point to a building? Why?
- What could be done to improve the matching process?

---

## 5. 🔥 Feature engineering: height, floors, GFA, heated volume (with source tracking)

---

For our energy estimation. We need:
- **Gross Floor Area (GFA, m²)**: proxy = footprint_area × number_of_floors
- **Gross Heated Volume (m³)**: proxy = footprint_area × height

##### Height estimation logic

Some data might be missing or incomplete. We need to handle these cases.
Most of our buildings will have a height from OBAT, but also a height from OSM. We need to prioritize which one to use and how to combine them. Because OBAT is an estimation based on 100m resolution data, we will prioritize the OSM height when available.

We will therefore compute: 
- IF OSM height is available, use it.
- IF OSM height is not available, use OBAT height.
- IF neither are available, and OSM has a `building:levels` attribute, use it. (we usually assume 3m per level)
- IF none are available, we will set the height to a default of 3m.

##### Source tracking

To keep the workflow transparent, we will also store **where each estimated value comes from** using `height_source`

This is useful later to:
- quantify uncertainty
- understand how much of the dataset depends on defaults
- explain limitations in the final brief

### 5.1 Height estimation

<div class="alert alert-danger">     

**🚀 TODO**

Add new building parameters to your dataframe.
1. define your default values
2. create the function `estimate_height()` to estimate the **building height in metres**.
3. apply the function to each row of the dataframe
4. Add the following columns to the building dataframe:
   - `height_m_est`
   - `height_source`
5. compute:
   - **GFA** (Gross Floor Area, in m²)
   - **Heated volume** (in m³)

In [ ]:
def estimate_height(row: pd.Series, default_levels: int, default_floor_height_m: float) -> int:
    """Estimate number of floors from available attributes.

    Priority:
    1) OSM height if present
    2) else OBAT height if present
    3) else OSM 'building:levels' * default_floor_height_m
    4) else default_levels * default_floor_height_m
    
    Returns
    -------
    pd.Series
        A series with:
        - height_m_est
        - height_source
    """

    return pd.Series({
        "height_m_est": height_m_est,
        "height_source": height_source,
    })



<div class="alert alert-info">

**💡 Tips:**

If you want to apply the same set of operations to all rows of a DataFrame, two common approaches are **[`apply(...)`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)** and **[`iterrows()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html)**.

- **`apply(..., axis=1)`** runs a function on each row and is well suited when you want to create one or more new columns from several existing columns
    ```python
    df[["newcol1,newcol2"]] = df.apply(method_to_apply, axis=1, additional_parameters) # the axis parameter specifies that we want to apply the function to each row
    ```

- **`iterrows()`** loops through the rows one by one. It is easier to understand, but it is usually slower and less efficient than `apply`.

- If you want to go back to the number of floor, we can keep the estimation: `height / default_floor_height_m`


### 5.2 Sanity checks, source summary, and quick visualisation

<div class="alert alert-danger">     

**🚀 TODO**

Add new building parameters to your dataframe.
1. Check that the results are coherent
- height must be positive
- GFA must be positive
- heated volume must be positive
2. Summarise the source columns
- how many buildings use OSM height
- how many use OBAT height
- how many use OSM levels
- how many rely on defaults
3. Produce one quick visualisation
- histogram of `height_m_est`,
- histogram of `gfa_m2`,
- histogram of `heated_volume_m3`,
- bar chart of `height_source`

<div class="alert alert-info">

**💡 Tips:**

- **[`assert`](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement)** is used to check that an important condition is true. It is useful for catching errors early. It will you stop the notebook immediately if something has gone wrong in your calculations.

    ```python
    assert (<VALUE> > 0).all(), "Some values are not positive" # the .all() method checks that all values are true
    ```

- `plot()` include the parameters `kind` that defines the type of chart
    - `hist`: histogram
    - `bar`: bar chart
    - `box`: box plot
    - `line`: line chart

---

## 6. 🏘️ Proxy energy model: simple TABULA archetypes

---

We do not know the real energy consumption of each building, so we have to build a **simple proxy model** for annual heating demand.

We will use TABULA typology system to get some coarse estimation, the rule of thumb we will apply is:  
1. **Older buildings** usually have higher heating demand.
2. **Compact residential blocks** usually perform better than small detached buildings

##### Simplified archetypes

We classify residential buildings into two broad groups:

- **`house_like`**  
  small footprint and low-rise buildings (SFH / TH)
- **`block_like`**  
  larger and/or taller residential buildings (MFH / AB)

This is a **proxy**, not a real building audit.


##### Step-by-step logic

1. assign a simple archetype from building size
2. map `(epoch, archetype)` to a specific demand in `kWh/m²a`
3. compute annual heating demand:

   `annual_heat_kwh = gfa_m2 × specific_demand_kwh_m2a`
4. convert to MWh (/ 1000)
5. rank buildings by annual heat demand

##### Why rank by annual heat demand?

We will rank by **absolute annual heat demand** (`annual_heat_mwh`).

This is a simple and defensible choice because:
- the client wants to identify buildings with the **largest heating burden**
- it is easy to interpret
- it avoids adding another complex scoring model

The drawback is that this favours **large buildings**.


### 6.1 Building archetypes

Create a **simple residential archetype** for each building.

A building is:
- `house_like` if it has **2 floors or fewer** and **small footprint** (e.g., < 300 m²)
- `block_like` otherwise

This is only a proxy to support the energy estimation.

<div class="alert alert-danger">

**🚀 TODO**

1. Create a function `classify_simple_archetype()`
2. Apply it to the dataframe
3. Add a new column: `simple_archetype`

In [ ]:
def classify_simple_archetype(row: pd.Series) -> str:
    """Classify a residential building into a simple archetype."""


    return builing_archetype

<div class="alert alert-info">

**💡 Tips**
 
- Use **`value_counts()`** to quickly check how many buildings fall into each archetype.

<div class="alert alert-success"> 

**🧠 Questions**:
- What is the limitation of reducing residential buildings to only two archetypes?

### 6.2 Lookup and energy estimation



<div class="alert alert-danger">

**🚀 TODO**

1. create a lookup table for `house_like` and `block_like`
   epoch codes from OBAT:
   - 1 = before 1980
   - 2 = 1980–1990
   - 3 = 1990–2000
   - 4 = 2000–2010
   - 5 = 2010–2020
2. write a function `get_specific_demand()`
3. apply it to the dataframe
4. compute:
   - `specific_demand_kwh_m2a`
   - `annual_heat_kwh`
   - `annual_heat_mwh`

In [ ]:
def get_specific_demand(row: pd.Series) -> float:
    """Return the specific heating demand from archetype and epoch."""

    return lookup_up_value


<div class="alert alert-info">

**💡 Tips**

- A `lookup table` is a simple table (often a dictionary) that maps one value to another, like an ID to a label (for example, 1 → "Residential", 2 → "Commercial"). It lets your code or dataset "look up" the meaning of a code instead of repeating the full text everywhere, which keeps things consistent, easier to change, and easier to query.

    For the exercise you will need
```json
    look_up = {
        "house_like": {
            "epoch1": <expected_consumption>,
            "epoch2": <expected_consumption>,
        },
        "block_like": {
            "epoch1": <expected_consumption>,
            "epoch2": <expected_consumption>,
        },
    }
```

- To get safely values from a dictionary, you can use: `.get()`. It returns a value for a key if it exists, or a default (like None or "missing") instead of crashing with KeyError.

```python
    params = {"crs": 4326, "buffer": 500}
    print(params["missing"])     # ❌ KeyError! 
    print(params.get("missing")) # ✅ None
    print(params.get("missing", "default")) # ✅ "default"
```

### 6.3 Rank retrofit candidates

We now rank buildings to identify the **top retrofit candidates**.

### Ranking rule

- **annual heating demand** (`annual_heat_mwh`)


<div class="alert alert-danger">

**🚀 TODO**

1. sort the residential buildings by `annual_heat_mwh`
2. create a ranking table
3. show the top 10 buildings
4. visualise the result with one simple chart

<div class="alert alert-info">

**💡 Tips**
- use **`sort_values()`** to rank buildings from highest to lowest energy demand.  
- a **bar chart** is useful to compare the top candidates directly.

<div class="alert alert-success"> 

**🧠 Questions**:
- Which buildings appear in the top 10, and why?
- What would change if you ranked by `kWh/m²a` instead of `annual_heat_mwh`?

---

## 7. 📊 Interactive visualization with Plotly

---

Until now, we have used static plots. These are useful for quick checks, but they do not let the user explore the data in detail.

In this section, we use **Plotly**, a Python library for creating **interactive visualizations**.

##### What is Plotly?

[Plotly](https://plotly.com/python/) is a visualization library that allows you to create charts and maps that users can interact with directly.

For example, interactivity can allow the user to:
- move around the figure
- zoom in and out
- hover over an element to see more information
- inspect individual buildings more easily

This is especially useful in spatial data science, where we often want to explore data at different scales and compare individual features.

##### Why use Plotly here?

Our goal is not only to compute building energy demand, but also to **communicate** the results clearly.

A static map shows the general pattern.  
An interactive map allows the user to:
- inspect a single building
- compare neighbouring buildings
- explore the relationship between geometry and estimated energy demand

### 7.1 Recreate one earlier chart with Plotly

Before building the map, let's recreate one of your earlier results with Plotly.

For example:
- the top 10 retrofit candidates,
- the distribution of estimated heights
- the count of buildings by height source

This helps you compare:
- **static plotting** (quick and simple)
- **interactive plotting** (better for exploration)

<div class="alert alert-danger">

**🚀 TODO**

1. Choose one graph you already created earlier in the notebook
2. Recreate it with Plotly
3. Compare the Plotly version with the static version

<div class="alert alert-info">

**💡 Tips**
- use **[`plotly.express.bar()`](https://plotly.com/python/bar-charts/)** or **[`plotly.express.histogram()`](https://plotly.com/python/histograms/)**, both methods take a simple dataframe as input

### 7.2 Create an interactive choropleth map

Now create an interactive map where building geometries are coloured by your chosen energy metric.

Recommended metric:
- `annual_heat_mwh`

A **[choropleth map](https://datavizcatalogue.com/methods/choropleth.html)** is a type of thematic map where features are coloured according to a value.

Here, each building polygon will be coloured by its estimated annual heating demand.

<div class="alert alert-danger">

**🚀 TODO**

Create an interactive choropleth map where:
1. building polygons are coloured by your chosen metric (for example `annual_heat_mwh`)
2. (optional) clean your data before visualizing
3. export the final graph or map to the `outputs/` folder

<div class="alert alert-info">

**💡 Tips**
- use **[`plotly.express.choropleth`](https://plotly.com/python/choropleth-maps/)**
- export your figure with plotly is very easy if you want the dynamic version you can use [`fig.write_html()`](https://plotly.com/python/interactive-html-export/), for a static version you can use [`fig.write_image()`](https://plotly.com/python/static-image-export/)

<div class="alert alert-success"> 

**🧠 Questions**:
- What does interactivity allow you to do that a static figure does not?
- Which type of result benefits more from interactivity: a simple bar chart or a building map? Why?
- When might a static figure still be preferable to an interactive one?

### 7.3 🥊 (optional) 3D exploration

Plotly does not easily extrude polygons in a free map view without extra complexity.
Instead, we can produce an interactive **3D scatter** of building centroids:
- x, y = projected coordinates (metres), you can use the centroid of the geometry
- z = `height_m_est`
- color = `building_usage`
This is not a map, but it helps explore the vertical structure! 

You can find some information about a 3D scatter plot with plotly, [here](https://plotly.com/python/3d-scatter-plots/).

## 8. Advisory brief (<500 words)

Write your brief below as Markdown. It must include:

1) **Top 3 retrofit candidates**  
- give building identifiers (your internal IDs) and map evidence (numbers)
- explain your ranking logic

2) **Technical risks** (at least 3)  
Examples: join error, missing attributes and imputation, CRS/unit errors, temporal mismatch, model validity.

3) **Ethical / governance risks** (at least 2)  
Examples: data bias/coverage, accountability, overconfidence in proxy maps, feedback loops in investment decisions.

4) **Licence/provenance note**  
- acknowledge OSM’s ODbL and the use of GHS-OBAT.



### Your brief

**Top 3 buildings and rationale**

- 1) ...
- 2) ...
- 3) ...

**Technical risks**

- ...
- ...

**Ethical / governance risks**

- ...
- ...

**Licence & provenance**

- ...


<div class="alert alert-success"> 

**🧠 Questions**:
- What is the single most fragile step in this pipeline, and what would you do to make it more robust?
- If the Sustainability Board treated your map as "truth", what is the most likely *bad* decision they might make?
- How would your results change if you doubled the assumed floor height (3 m → 6 m) in missing-data cases?

## 9. 👀 Peer Feedback 

Exchange your notebook with a classmate, compare your implementation and results, and discuss any differences, bugs, or unclear choices. After the discussion, improve your notebook based on the feedback you received.

#### Congratulations! 🎉 You have successfully completed the exercise.